# Function CNN — 2D & 3D Graph Classifier
Multi-task CNN that classifies function plots (2D curves and 3D surfaces), detects properties, and reconstructs images.

In [ ]:
import random
from io import BytesIO

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from torch.utils.data import DataLoader, Dataset

%matplotlib inline

print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# ── Tensor core & parallel processing setup ──
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision('high')
    except AttributeError:
        pass
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"cuDNN benchmark: enabled")
    print(f"TF32: enabled")

In [ ]:
FUNCTION_TYPES_2D = [
    'linear', 'quadratic', 'cubic', 'sine', 'cosine',
    'exponential', 'logarithmic', 'absolute',
]

FUNCTION_TYPES_3D = [
    'paraboloid', 'saddle', 'sine_surface', 'gaussian',
    'ripple', 'cone', 'hyperboloid', 'spiral_surface',
]

FUNCTION_TYPES = FUNCTION_TYPES_2D + FUNCTION_TYPES_3D
NUM_CLASSES = len(FUNCTION_TYPES)

FEATURE_NAMES = [
    'is_periodic',
    'is_monotone_increasing',
    'is_monotone_decreasing',
    'has_multiple_peaks',
    'is_bounded_above',
    'is_bounded_below',
    'is_3d',
    'is_symmetric',
    'has_saddle_point',
]
NUM_FEATURES = len(FEATURE_NAMES)

IMG_SIZE    = 128
BATCH_SIZE  = 64
NUM_EPOCHS  = 30
NUM_TRAIN   = 8000
NUM_VAL     = 1600
LR          = 5e-4
MODEL_PATH  = 'function_cnn.pth'

USE_AMP = torch.cuda.is_available()
DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Device: {DEVICE}")
print(f"Mixed precision: {USE_AMP}")
print(f"Classes: {NUM_CLASSES} ({len(FUNCTION_TYPES_2D)} 2D + {len(FUNCTION_TYPES_3D)} 3D)")

## Data Generation

In [ ]:
def generate_function_2d(func_type, x):
    features = np.zeros(NUM_FEATURES, dtype=np.float32)

    if func_type == 'linear':
        a = random.uniform(-3, 3)
        b = random.uniform(-5, 5)
        y = a * x + b
        if a > 0.1:   features[1] = 1
        elif a < -0.1: features[2] = 1

    elif func_type == 'quadratic':
        a = random.choice([-1, 1]) * random.uniform(0.5, 3)
        b = random.uniform(-3, 3)
        c = random.uniform(-5, 5)
        y = a * x**2 + b * x + c
        if a > 0: features[5] = 1
        else:     features[4] = 1
        features[7] = 1

    elif func_type == 'cubic':
        a = random.choice([-1, 1]) * random.uniform(0.1, 1)
        b = random.uniform(-2, 2)
        c = random.uniform(-3, 3)
        d = random.uniform(-5, 5)
        y = a * x**3 + b * x**2 + c * x + d
        features[3] = 1

    elif func_type == 'sine':
        a = random.uniform(1, 4)
        b = random.uniform(0.5, 3)
        c = random.uniform(0, 2 * np.pi)
        y = a * np.sin(b * x + c)
        features[[0, 3, 4, 5]] = 1

    elif func_type == 'cosine':
        a = random.uniform(1, 4)
        b = random.uniform(0.5, 3)
        c = random.uniform(0, 2 * np.pi)
        y = a * np.cos(b * x + c)
        features[[0, 3, 4, 5]] = 1

    elif func_type == 'exponential':
        a = random.uniform(0.5, 2)
        b = random.choice([-1, 1]) * random.uniform(0.2, 1)
        y = a * np.exp(b * x)
        if b > 0: features[1] = 1
        else:     features[2] = 1
        features[5] = 1

    elif func_type == 'logarithmic':
        a = random.choice([-1, 1]) * random.uniform(0.5, 3)
        b = random.uniform(-3, 3)
        y = a * np.log(np.abs(x) + 1) + b
        if a > 0: features[1] = 1
        else:     features[2] = 1

    elif func_type == 'absolute':
        a = random.choice([-1, 1]) * random.uniform(0.5, 3)
        b = random.uniform(-3, 3)
        c = random.uniform(-5, 5)
        y = a * np.abs(x + b) + c
        if a > 0: features[5] = 1
        else:     features[4] = 1
        features[7] = 1

    y = np.clip(y, -15, 15)
    return y.astype(np.float32), features

In [ ]:
def generate_function_3d(func_type, X, Y):
    features = np.zeros(NUM_FEATURES, dtype=np.float32)
    features[6] = 1

    if func_type == 'paraboloid':
        a = random.choice([-1, 1]) * random.uniform(0.3, 2)
        b = random.choice([-1, 1]) * random.uniform(0.3, 2)
        cx = random.uniform(-1, 1)
        cy = random.uniform(-1, 1)
        Z = a * (X - cx)**2 + b * (Y - cy)**2
        if a > 0 and b > 0:
            features[5] = 1
            features[7] = 1
        elif a < 0 and b < 0:
            features[4] = 1
            features[7] = 1

    elif func_type == 'saddle':
        a = random.uniform(0.3, 2)
        b = random.uniform(0.3, 2)
        Z = a * X**2 - b * Y**2
        features[8] = 1
        features[7] = 1

    elif func_type == 'sine_surface':
        a = random.uniform(1, 3)
        bx = random.uniform(0.5, 2)
        by = random.uniform(0.5, 2)
        Z = a * np.sin(bx * X) * np.cos(by * Y)
        features[[0, 3, 4, 5, 7]] = 1

    elif func_type == 'gaussian':
        a = random.uniform(1, 4)
        sx = random.uniform(0.5, 2)
        sy = random.uniform(0.5, 2)
        cx = random.uniform(-1, 1)
        cy = random.uniform(-1, 1)
        Z = a * np.exp(-((X - cx)**2 / (2 * sx**2) + (Y - cy)**2 / (2 * sy**2)))
        features[[4, 5, 7]] = 1

    elif func_type == 'ripple':
        a = random.uniform(1, 3)
        freq = random.uniform(1, 3)
        R = np.sqrt(X**2 + Y**2) + 1e-6
        Z = a * np.sin(freq * R) / R
        features[[0, 3, 4, 5, 7]] = 1

    elif func_type == 'cone':
        a = random.choice([-1, 1]) * random.uniform(0.5, 2)
        cx = random.uniform(-1, 1)
        cy = random.uniform(-1, 1)
        Z = a * np.sqrt((X - cx)**2 + (Y - cy)**2)
        if a > 0: features[1] = 1
        else:     features[2] = 1
        features[7] = 1

    elif func_type == 'hyperboloid':
        a = random.uniform(0.3, 1.5)
        b = random.uniform(0.3, 1.5)
        c = random.uniform(0.5, 2)
        Z = c * np.sqrt(1 + (X / a)**2 + (Y / b)**2)
        features[[1, 5, 7]] = 1

    elif func_type == 'spiral_surface':
        R = np.sqrt(X**2 + Y**2) + 1e-6
        theta = np.arctan2(Y, X)
        a = random.uniform(0.5, 2)
        freq = random.uniform(1, 3)
        Z = a * np.sin(freq * R + theta)
        features[[0, 3, 4, 5]] = 1

    Z = np.clip(Z, -15, 15)
    return Z.astype(np.float32), features

## Plotting

In [ ]:
# ── 2D: pure-numpy rasterizer (no matplotlib, no PNG roundtrip) ──
def plot_to_image_2d(x, y, size=IMG_SIZE):
    img = np.ones((size, size), dtype=np.float32)
    xs = np.linspace(0, size - 1, len(x))
    ys = (1 - (y - (-12)) / 24.0) * (size - 1)
    ys = np.clip(ys, 0, size - 1)

    xi = xs.astype(np.int32)
    yi = ys.astype(np.int32)

    for k in range(len(xi) - 1):
        x0, y0 = xi[k], yi[k]
        x1, y1 = xi[k + 1], yi[k + 1]
        steps = max(abs(x1 - x0), abs(y1 - y0), 1)
        xx = np.linspace(x0, x1, steps + 1).astype(np.int32)
        yy = np.linspace(y0, y1, steps + 1).astype(np.int32)
        img[yy, xx] = 0.0
        if size >= 64:
            yy_t = np.clip(yy + 1, 0, size - 1)
            img[yy_t, xx] = 0.0

    mid = size // 2
    img[mid, :] = np.minimum(img[mid, :], 0.7)
    img[:, mid] = np.minimum(img[:, mid], 0.7)
    return img


# ── 3D: reuse a single matplotlib figure across all samples ──
_fig3d = None
_ax3d  = None

def _get_3d_fig(size):
    global _fig3d, _ax3d
    if _fig3d is None:
        _fig3d = plt.figure(figsize=(2, 2), dpi=size // 2)
        _ax3d  = _fig3d.add_subplot(111, projection='3d')
    return _fig3d, _ax3d


def _canvas_to_rgb(fig):
    """Read RGB from canvas — works across matplotlib versions."""
    canvas = fig.canvas
    canvas.draw()
    # matplotlib >= 3.9: buffer_rgba; older: tostring_rgb
    if hasattr(canvas, 'buffer_rgba'):
        buf = np.asarray(canvas.buffer_rgba())  # H x W x 4
        return buf[..., :3]
    elif hasattr(canvas, 'tostring_rgb'):
        w, h = canvas.get_width_height()
        return np.frombuffer(canvas.tostring_rgb(), dtype=np.uint8).reshape(h, w, 3)
    else:
        # Last resort: render via PIL
        from io import BytesIO
        bio = BytesIO()
        fig.savefig(bio, format='png')
        bio.seek(0)
        return np.array(Image.open(bio).convert('RGB'))


def plot_to_image_3d(X, Y, Z, size=IMG_SIZE, elev=None, azim=None):
    if elev is None:
        elev = random.uniform(20, 50)
    if azim is None:
        azim = random.uniform(20, 340)

    fig, ax = _get_3d_fig(size)
    ax.cla()
    ax.plot_surface(X, Y, Z, cmap='viridis', alpha=0.9,
                    rstride=3, cstride=3, linewidth=0, antialiased=False)
    ax.view_init(elev=elev, azim=azim)
    ax.set_xlim(X.min(), X.max())
    ax.set_ylim(Y.min(), Y.max())
    z_range = max(abs(Z.min()), abs(Z.max()), 1)
    ax.set_zlim(-z_range, z_range)
    ax.axis('off')
    fig.tight_layout(pad=0)

    rgb = _canvas_to_rgb(fig)
    h, w = rgb.shape[:2]
    gray = (0.299 * rgb[..., 0] + 0.587 * rgb[..., 1] + 0.114 * rgb[..., 2])

    if (h, w) != (size, size):
        gray = np.array(Image.fromarray(gray.astype(np.uint8)).resize((size, size)), dtype=np.float32)
    return gray.astype(np.float32) / 255.0

## Dataset

In [ ]:
class FunctionDataset(Dataset):
    def __init__(self, n=NUM_TRAIN, img_size=IMG_SIZE):
        x_1d = np.linspace(-5, 5, 200)
        grid = np.linspace(-3, 3, 60)
        X, Y = np.meshgrid(grid, grid)

        print(f"Generating {n} samples...")
        # Pre-allocate (avoids per-item list-append + np.array copy)
        imgs   = np.empty((n, img_size, img_size), dtype=np.float32)
        labels = np.empty(n, dtype=np.int64)
        feats  = np.empty((n, NUM_FEATURES), dtype=np.float32)

        for i in range(n):
            if i % 1000 == 0 and i > 0:
                print(f"  {i}/{n}")
            label_idx = i % NUM_CLASSES  # direct, no list lookup
            ft = FUNCTION_TYPES[label_idx]

            if label_idx < len(FUNCTION_TYPES_2D):
                y, f = generate_function_2d(ft, x_1d)
                imgs[i] = plot_to_image_2d(x_1d, y, img_size)
            else:
                Z, f = generate_function_3d(ft, X, Y)
                imgs[i] = plot_to_image_3d(X, Y, Z, img_size)

            labels[i] = label_idx
            feats[i]  = f

        self.imgs   = torch.from_numpy(imgs).unsqueeze(1).contiguous(memory_format=torch.channels_last)
        self.labels = torch.from_numpy(labels)
        self.feats  = torch.from_numpy(feats)

    def to(self, device):
        """Move full dataset to device once — eliminates per-batch CPU→GPU copy."""
        self.imgs   = self.imgs.to(device)
        self.labels = self.labels.to(device)
        self.feats  = self.feats.to(device)
        return self

    def __len__(self): return len(self.labels)

    def __getitem__(self, i):
        return self.imgs[i], self.labels[i], self.feats[i]

## Model (ResNet-style with SE Attention)

In [ ]:
import torch.nn.functional as F


class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        w = self.pool(x).view(b, c)
        w = self.fc(w).view(b, c, 1, 1)
        return x * w


class ResConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, drop=0.1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
        )
        self.se = SEBlock(out_ch)
        self.skip = nn.Conv2d(in_ch, out_ch, 1, bias=False) if in_ch != out_ch else nn.Identity()
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2)
        self.drop = nn.Dropout2d(drop)

    def forward(self, x):
        identity = self.skip(x)
        out = self.conv(x)
        out = self.se(out)
        out = self.relu(out + identity)
        return self.drop(self.pool(out))


def _dec_block(in_ch, out_ch):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(),
        nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(),
    )


class FunctionCNN(nn.Module):
    """
    Multi-task U-Net with three heads + homoscedastic uncertainty weighting.
    Encoder feature maps e1..e4 are routed laterally into the decoder so
    high-frequency edge information bypasses the global-pooled bottleneck.
    """
    def __init__(self):
        super().__init__()

        # ── Encoder (sequential blocks, but exposed for skip connections) ──
        # 128 -> 64 -> 32 -> 16 -> 8 -> 4
        self.enc1 = ResConvBlock(1,   32,  0.05)   # e1: 64x64x32
        self.enc2 = ResConvBlock(32,  64,  0.10)   # e2: 32x32x64
        self.enc3 = ResConvBlock(64,  128, 0.15)   # e3: 16x16x128
        self.enc4 = ResConvBlock(128, 256, 0.20)   # e4: 8x8x256
        self.enc5 = ResConvBlock(256, 512, 0.25)   # e5: 4x4x512  (bottleneck)

        # ── Classification / detection heads from pooled bottleneck ──
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.shared_fc = nn.Sequential(
            nn.Linear(512, 512), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.3),
        )
        self.classifier = nn.Linear(256, NUM_CLASSES)
        self.detector = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, NUM_FEATURES),
        )

        # ── U-Net decoder with lateral skip connections ──
        # No dec_fc — operate directly on the spatial bottleneck e5
        self.dec5 = _dec_block(512 + 256, 256)   # up(e5) ⊕ e4 -> 8x8
        self.dec4 = _dec_block(256 + 128, 128)   # up(d5) ⊕ e3 -> 16x16
        self.dec3 = _dec_block(128 + 64,  64)    # up(d4) ⊕ e2 -> 32x32
        self.dec2 = _dec_block(64  + 32,  32)    # up(d3) ⊕ e1 -> 64x64
        self.dec_out = nn.Conv2d(32, 1, 3, padding=1)  # 128x128 raw logits

        # ── Homoscedastic uncertainty: learnable log-variances per task ──
        # Loss: exp(-s_i) * L_i + s_i,  where s_i = log(sigma_i^2)
        # Initialised at 0 → sigma_i = 1 → equivalent to unweighted at start.
        self.log_var_cls   = nn.Parameter(torch.zeros(()))
        self.log_var_feat  = nn.Parameter(torch.zeros(()))
        self.log_var_recon = nn.Parameter(torch.zeros(()))

    @staticmethod
    def _up(t):
        return F.interpolate(t, scale_factor=2, mode='bilinear', align_corners=False)

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        e5 = self.enc5(e4)

        # Heads from pooled bottleneck
        pooled = self.global_pool(e5).flatten(1)
        shared = self.shared_fc(pooled)
        logits      = self.classifier(shared)
        feat_logits = self.detector(shared)

        # Decoder with skip connections (concatenation, channel-wise)
        d5 = self.dec5(torch.cat([self._up(e5), e4], dim=1))   # 8x8x256
        d4 = self.dec4(torch.cat([self._up(d5), e3], dim=1))   # 16x16x128
        d3 = self.dec3(torch.cat([self._up(d4), e2], dim=1))   # 32x32x64
        d2 = self.dec2(torch.cat([self._up(d3), e1], dim=1))   # 64x64x32
        recon_logits = self.dec_out(self._up(d2))              # 128x128x1

        return logits, feat_logits, recon_logits

## Training & Evaluation

In [ ]:
CLS_LOSS   = nn.CrossEntropyLoss()
FEAT_LOSS  = nn.BCEWithLogitsLoss()
RECON_LOSS = nn.BCEWithLogitsLoss()


def uncertainty_weighted_loss(L_cls, L_feat, L_recon, log_var_cls, log_var_feat, log_var_recon):
    """
    Kendall & Gal (2018) homoscedastic uncertainty.
    L = sum_i [ exp(-s_i) * L_i + s_i ],  s_i = log(sigma_i^2)
    Auto-balances task gradients; the +s_i term penalises blowing variances up.
    """
    return (torch.exp(-log_var_cls)   * L_cls   + log_var_cls
          + torch.exp(-log_var_feat)  * L_feat  + log_var_feat
          + torch.exp(-log_var_recon) * L_recon + log_var_recon)


def run_epoch(model, loader, optimizer, scaler=None, raw_model=None):
    model.train()
    total_loss = torch.zeros((), device=DEVICE)
    correct    = torch.zeros((), device=DEVICE, dtype=torch.long)
    total      = 0

    # Access learnable log-variances on the underlying (possibly-compiled) model
    rm = raw_model if raw_model is not None else model

    for imgs, labels, feats in loader:
        if imgs.device != DEVICE:
            imgs   = imgs.to(DEVICE, non_blocking=True, memory_format=torch.channels_last)
            labels = labels.to(DEVICE, non_blocking=True)
            feats  = feats.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        if scaler is not None:
            with torch.amp.autocast('cuda'):
                logits, feat_logits, recon_logits = model(imgs)
                L_cls   = CLS_LOSS(logits, labels)
                L_feat  = FEAT_LOSS(feat_logits, feats)
                L_recon = RECON_LOSS(recon_logits, imgs)
                loss = uncertainty_weighted_loss(
                    L_cls, L_feat, L_recon,
                    rm.log_var_cls, rm.log_var_feat, rm.log_var_recon,
                )
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits, feat_logits, recon_logits = model(imgs)
            L_cls   = CLS_LOSS(logits, labels)
            L_feat  = FEAT_LOSS(feat_logits, feats)
            L_recon = RECON_LOSS(recon_logits, imgs)
            loss = uncertainty_weighted_loss(
                L_cls, L_feat, L_recon,
                rm.log_var_cls, rm.log_var_feat, rm.log_var_recon,
            )
            loss.backward()
            optimizer.step()

        total_loss += loss.detach()
        correct    += (logits.argmax(1) == labels).sum()
        total      += labels.size(0)

    return (total_loss / len(loader)).item(), (correct.item() / total)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct = torch.zeros((), device=DEVICE, dtype=torch.long)
    total   = 0
    for imgs, labels, _ in loader:
        if imgs.device != DEVICE:
            imgs   = imgs.to(DEVICE, non_blocking=True, memory_format=torch.channels_last)
            labels = labels.to(DEVICE, non_blocking=True)
        if USE_AMP:
            with torch.amp.autocast('cuda'):
                logits, _, _ = model(imgs)
        else:
            logits, _, _ = model(imgs)
        correct += (logits.argmax(1) == labels).sum()
        total   += labels.size(0)
    return correct.item() / total

## Generate Data

In [ ]:
train_set = FunctionDataset(NUM_TRAIN)
val_set   = FunctionDataset(NUM_VAL)

# Move full datasets to GPU once — eliminates per-batch CPU→GPU copy entirely
if torch.cuda.is_available():
    train_set.to(DEVICE)
    val_set.to(DEVICE)
    print(f"Datasets resident on {DEVICE}")

# pin_memory and num_workers are irrelevant when data is already on the GPU
train_loader = DataLoader(train_set, BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_set,   BATCH_SIZE, shuffle=False, num_workers=0)

## Train

In [ ]:
model = FunctionCNN().to(DEVICE, memory_format=torch.channels_last)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Memory format: channels_last (NHWC — optimal for tensor cores)")

raw_model = model  # keep reference for state_dict + log_var access
try:
    model = torch.compile(model)
    print("torch.compile: enabled")
except Exception as e:
    print(f"torch.compile: not available ({e})")

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
scaler    = torch.amp.GradScaler('cuda') if USE_AMP else None

best_val = 0.0
history  = {'loss': [], 'train_acc': [], 'val_acc': [],
            'w_cls': [], 'w_feat': [], 'w_recon': []}

for epoch in range(1, NUM_EPOCHS + 1):
    loss, train_acc = run_epoch(model, train_loader, optimizer, scaler, raw_model=raw_model)
    val_acc         = evaluate(model, val_loader)
    scheduler.step()

    # Effective task weights: w_i = exp(-s_i) = 1 / sigma_i^2
    with torch.no_grad():
        w_cls   = torch.exp(-raw_model.log_var_cls).item()
        w_feat  = torch.exp(-raw_model.log_var_feat).item()
        w_recon = torch.exp(-raw_model.log_var_recon).item()

    history['loss'].append(loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['w_cls'].append(w_cls)
    history['w_feat'].append(w_feat)
    history['w_recon'].append(w_recon)

    if val_acc > best_val:
        best_val = val_acc
        torch.save(raw_model.state_dict(), MODEL_PATH)

    print(f"  [{epoch:2d}/{NUM_EPOCHS}]  loss={loss:.4f}  "
          f"train={train_acc*100:.1f}%  val={val_acc*100:.1f}%  "
          f"w(cls/feat/recon)={w_cls:.2f}/{w_feat:.2f}/{w_recon:.2f}")

print(f"\nBest val accuracy: {best_val*100:.1f}%")
raw_model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True))

## Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(history['loss']);  ax1.set_title('Loss');     ax1.set_xlabel('Epoch')
ax2.plot(history['train_acc'], label='train')
ax2.plot(history['val_acc'],   label='val')
ax2.set_title('Accuracy'); ax2.set_xlabel('Epoch'); ax2.legend()
plt.tight_layout()
plt.show()

## Predictions Visualization

In [ ]:
model.eval()
n_show = 8
idx = random.sample(range(len(val_set)), n_show)

# Batched single forward pass (was n_show separate passes with batch=1)
batch = torch.stack([val_set[i][0] for i in idx]).to(
    DEVICE, non_blocking=True, memory_format=torch.channels_last)
labels_show = torch.stack([val_set[i][1] for i in idx])

with torch.no_grad():
    if USE_AMP:
        with torch.amp.autocast('cuda'):
            logits, _, recon_logits = model(batch)
    else:
        logits, _, recon_logits = model(batch)
    preds = logits.argmax(1).cpu()
    recons = torch.sigmoid(recon_logits).cpu()  # apply sigmoid for visualization

fig, axes = plt.subplots(3, n_show, figsize=(n_show * 2, 6))
for col in range(n_show):
    img  = batch[col, 0].cpu()
    lbl  = labels_show[col].item()
    pred = preds[col].item()
    color = 'green' if pred == lbl else 'red'

    axes[0, col].imshow(img, cmap='gray')
    axes[0, col].set_title(FUNCTION_TYPES[lbl][:6], fontsize=7)
    axes[0, col].axis('off')

    axes[1, col].imshow(img, cmap='gray')
    axes[1, col].set_title(FUNCTION_TYPES[pred][:6], fontsize=7, color=color)
    axes[1, col].axis('off')

    axes[2, col].imshow(recons[col, 0], cmap='gray')
    axes[2, col].set_title('recon', fontsize=7)
    axes[2, col].axis('off')

for row, lbl in enumerate(['Original', 'Predicted', 'Reconstructed']):
    axes[row, 0].set_ylabel(lbl, fontsize=8)
plt.tight_layout()
plt.show()

## Demo — Analyze Each Function Type

In [ ]:
@torch.no_grad()
def analyze(func_type):
    model.eval()

    if func_type in FUNCTION_TYPES_3D:
        grid = np.linspace(-3, 3, 60)
        X, Y = np.meshgrid(grid, grid)
        Z, true_feats = generate_function_3d(func_type, X, Y)
        img = plot_to_image_3d(X, Y, Z)
    else:
        x = np.linspace(-5, 5, 200)
        y, true_feats = generate_function_2d(func_type, x)
        img = plot_to_image_2d(x, y)

    t = torch.from_numpy(img).unsqueeze(0).unsqueeze(0)
    t = t.to(DEVICE, non_blocking=True, memory_format=torch.channels_last)
    if USE_AMP:
        with torch.amp.autocast('cuda'):
            logits, feat_logits, _ = model(t)
    else:
        logits, feat_logits, _ = model(t)
    probs     = torch.softmax(logits, 1).cpu().numpy()[0]
    pred_feat = torch.sigmoid(feat_logits).cpu().numpy()[0]  # logits -> probabilities
    pred_cls  = probs.argmax()

    dim_label = "3D" if func_type in FUNCTION_TYPES_3D else "2D"
    print(f"\n{'=' * 52}")
    print(f"  FUNCTION ANALYSIS ({dim_label})")
    print(f"{'=' * 52}")
    print(f"  True type : {func_type}")
    print(f"  Predicted : {FUNCTION_TYPES[pred_cls]}  ({probs[pred_cls]*100:.1f}% confidence)")
    print(f"\n  Top-3 predictions:")
    for i in probs.argsort()[::-1][:3]:
        print(f"    {FUNCTION_TYPES[i]:20s}  {probs[i]*100:.1f}%")
    print(f"\n  Detected properties:")
    for name, val in zip(FEATURE_NAMES, pred_feat):
        marker = "Y" if val > 0.5 else "."
        print(f"    {marker} {name:30s} ({val:.2f})")

for ft in FUNCTION_TYPES:
    analyze(ft)

## Baseline comparison: 1-layer CNN vs FunctionCNN

To understand what the architecture is actually buying we train a deliberately
minimal 1-conv-layer CNN on the same data and training schedule, then compare
validation accuracy side-by-side. Spoiler: the deeper, multi-task model
substantially outperforms the baseline — exactly as it should.

In [ ]:
import torch.nn as nn
import torch
import torch.optim as optim
from torch.utils.data import DataLoader

class BaselineCNN(nn.Module):
    """1-conv-layer CNN. Same forward signature as FunctionCNN for drop-in use."""
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(4),
            nn.AdaptiveAvgPool2d(8),
        )
        self.classifier    = nn.Linear(16 * 8 * 8, NUM_CLASSES)
        self.detector_head = nn.Sequential(nn.Linear(16 * 8 * 8, NUM_FEATURES), nn.Sigmoid())

    def forward(self, x):
        z = self.features(x).flatten(1)
        return self.classifier(z), self.detector_head(z), x  # recon = identity

BASELINE_EPOCHS = 10  # baseline is small; short schedule is enough

baseline = BaselineCNN().to(device)
print(f'Baseline parameters: {sum(p.numel() for p in baseline.parameters()):,}')

opt   = optim.AdamW(baseline.parameters(), lr=1e-3, weight_decay=1e-3)
sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=BASELINE_EPOCHS)
scaler_b = torch.amp.GradScaler('cuda') if USE_AMP else None

best_baseline = 0.0
for epoch in range(1, BASELINE_EPOCHS + 1):
    loss, tr_acc = run_epoch(baseline, train_loader, opt, device, scaler_b)
    val_acc      = evaluate(baseline, val_loader, device)
    sched.step()
    if val_acc > best_baseline:
        best_baseline = val_acc
    print(f'  [baseline {epoch:2d}/{BASELINE_EPOCHS}] loss={loss:.4f} train={tr_acc*100:.1f}% val={val_acc*100:.1f}%')

print(f'\nBaseline best val acc:    {best_baseline*100:.1f}%')
print(f'FunctionCNN best val acc: {best_val*100:.1f}%')
print(f'Gain from deeper + multi-task: +{(best_val - best_baseline)*100:.1f} pp')
